# Orquestrador - Pedidos com garantias de entrega, evolução de esquema e logs

Este caderno é o **painel de controle** da atividade. Ele **não** contém a lógica dos serviços:
sobe o ambiente, inicia os serviços (`servicos/*.py`) como processos de verdade, publica os lotes,
injeta falhas, acompanha o *lag*, grava a saída com o `escritor_saida.py` (fornecido no kit) e roda o `verificar.py`.

**Convenções que os seus serviços precisam seguir**

| Item | Convenção |
| --- | --- |
| Endereços | variáveis de ambiente `KAFKA_BOOTSTRAP` (padrão `localhost:9094`) e `SCHEMA_REGISTRY_URL` (padrão `http://localhost:8081`) |
| `group.id` | `grp-pedidos`, `grp-estoque`, `grp-pagamento`, `grp-metricas` |
| Processos | um arquivo por serviço: `servicos/pedidos.py`, `servicos/estoque.py`, `servicos/pagamento.py`, `servicos/metricas.py` |
| Estoque final | `python servicos/estoque.py --exportar` grava `saida/estoque_final.csv` |
| Métricas | `python servicos/metricas.py --exportar` grava `saida/metricas.csv` |
| Saída dos pedidos e dos logs | `python escritor_saida.py` (fornecido) grava os demais arquivos de `saida/` a partir dos tópicos |

Execute as células **em ordem**. A etapa 5 só deve ser concluída depois que você alterar os serviços para a v2.

In [1]:
import json
import os
import signal
import subprocess
import sys
import time
import urllib.error
import urllib.request
from pathlib import Path

BOOTSTRAP = os.environ.get("KAFKA_BOOTSTRAP", "localhost:9094")
REGISTRY = os.environ.get("SCHEMA_REGISTRY_URL", "http://localhost:8081")

RAIZ = Path.cwd()
PASTA_LOGS = RAIZ / "logs"
PASTA_LOGS.mkdir(exist_ok=True)

# serviço -> (arquivo, group.id, tópicos que consome)
SERVICOS = {
    "pedidos":  ("servicos/pedidos.py", "grp-pedidos", ["pedidos"]),
    "estoque":  ("servicos/estoque.py", "grp-estoque",
                 ["estoque-inicial", "pedidos-validados", "pedidos-recusados"]),
    "pagamento": ("servicos/pagamento.py", "grp-pagamento", ["clientes", "estoque-reservado"]),
    "metricas": ("servicos/metricas.py", "grp-metricas", ["logs-eventos"]),
}
# subjects do PedidoEvento (evoluem de v1 para v2) e do LogEvento (fica na v1)
SUBJECTS = ["pedidos-validados-value", "estoque-reservado-value",
            "pedidos-confirmados-value", "pedidos-recusados-value"]
SUBJECT_LOG = "logs-eventos-value"

processos = {}
parados_por_nos = set()      # serviços encerrados de propósito (não são "erro")
print("Kafka:", BOOTSTRAP, "| Schema Registry:", REGISTRY)

Kafka: localhost:9094 | Schema Registry: http://localhost:8081


## Funções de apoio

In [2]:
def iniciar(nome):
    """Inicia um serviço como processo independente; o console vai para logs/<nome>.log."""
    if nome in processos and processos[nome].poll() is None:
        print(f"{nome}: já está rodando (pid {processos[nome].pid})")
        return
    parados_por_nos.discard(nome)
    arquivo = SERVICOS[nome][0]
    log = open(PASTA_LOGS / f"{nome}.log", "a", encoding="utf-8")
    env = dict(os.environ, KAFKA_BOOTSTRAP=BOOTSTRAP, SCHEMA_REGISTRY_URL=REGISTRY,
               PYTHONUNBUFFERED="1")
    processos[nome] = subprocess.Popen([sys.executable, arquivo], stdout=log,
                                       stderr=subprocess.STDOUT, env=env, cwd=RAIZ)
    print(f"{nome}: iniciado (pid {processos[nome].pid})")


def parar(nome, forcar=False):
    """Encerra um serviço. forcar=True simula uma queda (kill), sem fechar consumidor nem commit."""
    p = processos.get(nome)
    if p is None or p.poll() is not None:
        print(f"{nome}: não está rodando")
        return
    parados_por_nos.add(nome)
    if forcar:
        p.kill()
        print(f"{nome}: MORTO à força (pid {p.pid})")
    else:
        p.send_signal(signal.SIGINT if os.name != "nt" else signal.CTRL_BREAK_EVENT)
        try:
            p.wait(timeout=15)
            print(f"{nome}: encerrado normalmente")
        except subprocess.TimeoutExpired:
            p.kill()
            print(f"{nome}: não respondeu, encerrado à força")
    p.wait()


def status():
    for nome in SERVICOS:
        p = processos.get(nome)
        estado = "parado" if p is None or p.poll() is not None else f"rodando (pid {p.pid})"
        if p is not None and p.poll() not in (None, 0) and nome not in parados_por_nos:
            estado = f"ENCERROU COM ERRO (código {p.poll()}) - veja logs/{nome}.log"
        print(f"{nome:10s} {estado}")


def ver_log(nome, linhas=15):
    caminho = PASTA_LOGS / f"{nome}.log"
    if caminho.exists():
        print("\n".join(caminho.read_text(encoding="utf-8", errors="replace").splitlines()[-linhas:]))


def ingestar(arquivo, topico, chave, extra=()):
    """Publica um CSV com o ingestao.py fornecido."""
    cmd = [sys.executable, "ingestao.py", "--arquivo", arquivo, "--topico", topico,
           "--chave", chave, "--bootstrap", BOOTSTRAP, *extra]
    r = subprocess.run(cmd, capture_output=True, text=True, cwd=RAIZ)
    print(r.stdout[-900:] or r.stderr[-900:])
    if r.returncode != 0:
        raise RuntimeError("ingestao.py terminou com erro")

In [3]:
from confluent_kafka import Consumer, TopicPartition
from confluent_kafka.admin import AdminClient


def lag_do_grupo(grupo, topicos):
    """Soma de (fim do log - offset confirmado) das partições dos tópicos para o grupo."""
    consumidor = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": grupo,
                           "enable.auto.commit": False})
    meta = consumidor.list_topics(timeout=10)
    total = 0
    for t in topicos:
        if t not in meta.topics:
            continue
        pars = [TopicPartition(t, p) for p in meta.topics[t].partitions]
        for tp in consumidor.committed(pars, timeout=10):
            baixo, alto = consumidor.get_watermark_offsets(tp, timeout=10)
            confirmado = tp.offset if tp.offset >= 0 else baixo
            total += max(0, alto - confirmado)
    consumidor.close()
    return total


def esperar_lag_zero(timeout=120, estavel=3, margem=4):
    """Espera os grupos esvaziarem: lag total <= `margem` e sem mudar por `estavel` leituras seguidas.
    A margem existe porque transações gravam marcadores de controle no log; eles ocupam offsets
    e aparecem como um lag residual de 1 que nunca desaparece."""
    limite, ok, anterior = time.time() + timeout, 0, None
    while time.time() < limite:
        lags = {n: lag_do_grupo(g, t) for n, (_, g, t) in SERVICOS.items()}
        print(time.strftime("%H:%M:%S"), lags)
        ok = ok + 1 if (sum(lags.values()) <= margem and lags == anterior) else 0
        anterior = lags
        if ok >= estavel:
            print("Grupos em dia.")
            return True
        time.sleep(3)
    print("Tempo esgotado: algum serviço não terminou. Use status() e ver_log().")
    return False


def registry(metodo, caminho, corpo=None):
    dados = json.dumps(corpo).encode() if corpo is not None else None
    req = urllib.request.Request(REGISTRY + caminho, data=dados, method=metodo,
                                 headers={"Content-Type": "application/vnd.schemaregistry.v1+json"})
    try:
        with urllib.request.urlopen(req, timeout=10) as r:
            return r.status, json.loads(r.read() or b"null")
    except urllib.error.HTTPError as e:
        return e.code, json.loads(e.read() or b"null")


def registrar(subject, arquivo_avsc):
    """Registra o schema do arquivo no subject. Devolve (status HTTP, resposta)."""
    schema = Path(arquivo_avsc).read_text(encoding="utf-8")
    return registry("POST", f"/subjects/{subject}/versions", {"schema": schema})

## Etapa 1 - Ambiente (Kafka e Schema Registry)

In [4]:
# começa de um Kafka limpo
subprocess.run(["docker", "compose", "down", "-v"], capture_output=True, text=True, cwd=RAIZ)
r = subprocess.run(["docker", "compose", "up", "-d"], capture_output=True, text=True, cwd=RAIZ)
print(r.stdout or r.stderr)

for _ in range(40):
    try:
        AdminClient({"bootstrap.servers": BOOTSTRAP}).list_topics(timeout=3)
        codigo, _ = registry("GET", "/subjects")
        if codigo == 200:
            print("Kafka e Schema Registry no ar.")
            break
    except Exception:
        pass
    time.sleep(3)
else:
    raise RuntimeError("Ambiente não respondeu a tempo. Veja `docker compose logs`.")

 Network kafka-pedidos_default Creating 
 Network kafka-pedidos_default Creating 
 Volume kafka-pedidos_kafka_data Creating 
 Volume kafka-pedidos_kafka_data Creating 
 Volume kafka-pedidos_kafka_data Created 
 Volume kafka-pedidos_kafka_data Created 
 Network kafka-pedidos_default Created 
 Network kafka-pedidos_default Created 
 Container kafka-pedidos Creating 
 Container kafka-pedidos Created 
 Container kafka-ui-pedidos Creating 
 Container schema-registry-pedidos Creating 
 Container kafka-ui-pedidos Created 
 Container schema-registry-pedidos Created 
 Container kafka-pedidos Starting 
 Container kafka-pedidos Started 
 Container schema-registry-pedidos Starting 
 Container kafka-ui-pedidos Starting 
 Container schema-registry-pedidos Started 
 Container kafka-ui-pedidos Started 



%6|1792795256.086|FAIL|rdkafka#producer-1| [thrd:localhost:9094/bootstrap]: localhost:9094/bootstrap: Disconnected: connection reset by peer (after 1ms in state APIVERSION_QUERY)
%6|1792795256.137|FAIL|rdkafka#producer-1| [thrd:localhost:9094/bootstrap]: localhost:9094/bootstrap: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)
%6|1792795256.189|FAIL|rdkafka#producer-1| [thrd:localhost:9094/bootstrap]: localhost:9094/bootstrap: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)
%6|1792795256.240|FAIL|rdkafka#producer-1| [thrd:localhost:9094/bootstrap]: localhost:9094/bootstrap: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)


%6|1792795256.291|FAIL|rdkafka#producer-1| [thrd:localhost:9094/bootstrap]: localhost:9094/bootstrap: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)
%6|1792795256.343|FAIL|rdkafka#producer-1| [thrd:localhost:9094/bootstrap]: localhost:9094/bootstrap: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)
%6|1792795256.394|FAIL|rdkafka#producer-1| [thrd:localhost:9094/bootstrap]: localhost:9094/bootstrap: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)
%6|1792795256.446|FAIL|rdkafka#producer-1| [thrd:localhost:9094/bootstrap]: localhost:9094/bootstrap: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)


%6|1792795256.497|FAIL|rdkafka#producer-1| [thrd:localhost:9094/bootstrap]: localhost:9094/bootstrap: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)
%6|1792795256.551|FAIL|rdkafka#producer-1| [thrd:localhost:9094/bootstrap]: localhost:9094/bootstrap: Disconnected: connection reset by peer (after 2ms in state APIVERSION_QUERY)
%6|1792795256.600|FAIL|rdkafka#producer-1| [thrd:localhost:9094/bootstrap]: localhost:9094/bootstrap: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)
%6|1792795256.651|FAIL|rdkafka#producer-1| [thrd:localhost:9094/bootstrap]: localhost:9094/bootstrap: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)


%6|1792795256.702|FAIL|rdkafka#producer-1| [thrd:localhost:9094/bootstrap]: localhost:9094/bootstrap: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)
%6|1792795256.754|FAIL|rdkafka#producer-1| [thrd:localhost:9094/bootstrap]: localhost:9094/bootstrap: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)
%6|1792795256.805|FAIL|rdkafka#producer-1| [thrd:localhost:9094/bootstrap]: localhost:9094/bootstrap: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)
%6|1792795256.856|FAIL|rdkafka#producer-1| [thrd:localhost:9094/bootstrap]: localhost:9094/bootstrap: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)


...

%6|1792795258.398|FAIL|rdkafka#producer-1| [thrd:localhost:9094/bootstrap]: localhost:9094/bootstrap: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)
%6|1792795258.449|FAIL|rdkafka#producer-1| [thrd:localhost:9094/bootstrap]: localhost:9094/bootstrap: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)
%6|1792795258.500|FAIL|rdkafka#producer-1| [thrd:localhost:9094/bootstrap]: localhost:9094/bootstrap: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)
%6|1792795258.551|FAIL|rdkafka#producer-1| [thrd:localhost:9094/bootstrap]: localhost:9094/bootstrap: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)


%6|1792795258.602|FAIL|rdkafka#producer-1| [thrd:localhost:9094/bootstrap]: localhost:9094/bootstrap: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)
%6|1792795258.655|FAIL|rdkafka#producer-1| [thrd:localhost:9094/bootstrap]: localhost:9094/bootstrap: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)
%6|1792795258.706|FAIL|rdkafka#producer-1| [thrd:localhost:9094/bootstrap]: localhost:9094/bootstrap: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)
%6|1792795258.757|FAIL|rdkafka#producer-1| [thrd:localhost:9094/bootstrap]: localhost:9094/bootstrap: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)


%6|1792795258.809|FAIL|rdkafka#producer-1| [thrd:localhost:9094/bootstrap]: localhost:9094/bootstrap: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)
%6|1792795258.861|FAIL|rdkafka#producer-1| [thrd:localhost:9094/bootstrap]: localhost:9094/bootstrap: Disconnected: connection reset by peer (after 1ms in state APIVERSION_QUERY)
%6|1792795258.914|FAIL|rdkafka#producer-1| [thrd:localhost:9094/bootstrap]: localhost:9094/bootstrap: Disconnected: connection reset by peer (after 2ms in state APIVERSION_QUERY)
%6|1792795258.963|FAIL|rdkafka#producer-1| [thrd:localhost:9094/bootstrap]: localhost:9094/bootstrap: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)


%6|1792795259.014|FAIL|rdkafka#producer-1| [thrd:localhost:9094/bootstrap]: localhost:9094/bootstrap: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)
%6|1792795259.067|FAIL|rdkafka#producer-1| [thrd:localhost:9094/bootstrap]: localhost:9094/bootstrap: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)


Kafka e Schema Registry no ar.


## Etapa 2 - Compatibilidade e schema v1

Define a compatibilidade **BACKWARD** e registra a v1 do `PedidoEvento` nos subjects dos eventos de pedido e a v1 do `LogEvento` no subject dos logs.
Se o seu serviço já registra o schema ao produzir, esta célula só confirma a configuração.

In [5]:
for s in SUBJECTS:
    print(s, registry("PUT", f"/config/{s}", {"compatibility": "BACKWARD"}))
    print("   v1 ->", registrar(s, "schemas/PedidoEvento.v1.avsc"))
print(SUBJECT_LOG, registry("PUT", f"/config/{SUBJECT_LOG}", {"compatibility": "BACKWARD"}))
print("   v1 ->", registrar(SUBJECT_LOG, "schemas/LogEvento.v1.avsc"))

pedidos-validados-value (200, {'compatibility': 'BACKWARD'})


   v1 -> (200, {'id': 1})
estoque-reservado-value (200, {'compatibility': 'BACKWARD'})
   v1 -> (200, {'id': 1})
pedidos-confirmados-value (200, {'compatibility': 'BACKWARD'})


   v1 -> (200, {'id': 1})
pedidos-recusados-value (200, {'compatibility': 'BACKWARD'})
   v1 -> (200, {'id': 1})
logs-eventos-value (200, {'compatibility': 'BACKWARD'})
   v1 -> (200, {'id': 2})


## Etapa 3 - Dados de apoio e serviços

In [6]:
import shutil
shutil.rmtree("saida", ignore_errors=True)       # começa de uma saída limpa
Path("saida").mkdir(exist_ok=True)
shutil.rmtree("estado", ignore_errors=True)      # estado dos serviços

ingestar("entrada/clientes.csv", "clientes", "cliente_id", ["--particoes", "1", "--numericos", "limite_pedido"])
ingestar("entrada/estoque_inicial.csv", "estoque-inicial", "sku", ["--particoes", "1", "--numericos", "preco,quantidade"])

for nome in ["metricas", "pagamento", "estoque", "pedidos"]:     # consumidores antes dos produtores
    iniciar(nome)
time.sleep(5)
status()

Tópico 'clientes' criado com 1 partição(ões).

--- Resumo da ingestão ---
Arquivo:            entrada/clientes.csv
Tópico:             clientes
Mensagens enviadas: 23
Confirmadas:        23
Falhas:             0
Por partição:       {0: 23}
Tempo:              0.19s



Tópico 'estoque-inicial' criado com 1 partição(ões).

--- Resumo da ingestão ---
Arquivo:            entrada/estoque_inicial.csv
Tópico:             estoque-inicial
Mensagens enviadas: 16
Confirmadas:        16
Falhas:             0
Por partição:       {0: 16}
Tempo:              0.12s

metricas: iniciado (pid 369690)
pagamento: iniciado (pid 369691)
estoque: iniciado (pid 369692)
pedidos: iniciado (pid 369693)


pedidos    rodando (pid 369693)
estoque    rodando (pid 369692)
pagamento  rodando (pid 369691)
metricas   rodando (pid 369690)


## Etapa 4 - Lote 1 e falha injetada

In [7]:
# em segundo plano e devagar, para a falha pegar o lote em andamento
ingestao_lote1 = subprocess.Popen(
    [sys.executable, "ingestao.py", "--arquivo", "entrada/pedidos_lote1.csv", "--topico", "pedidos",
     "--chave", "sku", "--bootstrap", BOOTSTRAP, "--campo-timestamp", "timestamp", "--intervalo-ms", "150"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=RAIZ)
time.sleep(4)

In [8]:
# FALHA INJETADA: derruba estoque, pagamento e metricas no meio do processamento e sobe de novo.
for nome in ["estoque", "pagamento", "metricas"]:
    parar(nome, forcar=True)
time.sleep(10)
for nome in ["estoque", "pagamento", "metricas"]:
    iniciar(nome)
time.sleep(5)
status()

saida_ingestao, _ = ingestao_lote1.communicate()
print(saida_ingestao[-900:])
if ingestao_lote1.returncode != 0:
    raise RuntimeError("ingestao.py terminou com erro")

estoque: MORTO à força (pid 369692)
pagamento: MORTO à força (pid 369691)
metricas: MORTO à força (pid 369690)


estoque: iniciado (pid 369774)
pagamento: iniciado (pid 369775)
metricas: iniciado (pid 369776)


pedidos    rodando (pid 369693)
estoque    rodando (pid 369774)
pagamento  rodando (pid 369775)
metricas   rodando (pid 369776)
/home/helo/kafka-pedidos/ingestao.py:185: DeprecationWarning: key_serializer does not implement kafka.serializer.Serializer
  main()
/home/helo/kafka-pedidos/ingestao.py:185: DeprecationWarning: value_serializer does not implement kafka.serializer.Serializer
  main()
Tópico 'pedidos' já existia (partições não alteradas).

--- Resumo da ingestão ---
Arquivo:            entrada/pedidos_lote1.csv
Tópico:             pedidos
Mensagens enviadas: 62
Confirmadas:        62
Falhas:             0
Sem chave ('sku' vazio): 2 (distribuídas sem garantia de ordem)
Por partição:       {0: 12, 1: 16, 2: 8, 3: 26}
Tempo:              9.45s



In [9]:
esperar_lag_zero()

19:41:34 {'pedidos': 0, 'estoque': 43, 'pagamento': 1, 'metricas': 46}


19:41:37 {'pedidos': 0, 'estoque': 43, 'pagamento': 1, 'metricas': 46}


19:41:41 {'pedidos': 0, 'estoque': 43, 'pagamento': 1, 'metricas': 46}


19:41:44 {'pedidos': 0, 'estoque': 43, 'pagamento': 1, 'metricas': 46}


19:41:47 {'pedidos': 0, 'estoque': 43, 'pagamento': 1, 'metricas': 46}


...

19:42:07 {'pedidos': 0, 'estoque': 0, 'pagamento': 0, 'metricas': 0}


19:42:10 {'pedidos': 0, 'estoque': 0, 'pagamento': 0, 'metricas': 0}


19:42:14 {'pedidos': 0, 'estoque': 0, 'pagamento': 0, 'metricas': 0}


19:42:17 {'pedidos': 0, 'estoque': 0, 'pagamento': 0, 'metricas': 0}
Grupos em dia.


True

## Etapa 5 - Evolução de esquema (v1 -> v2)

1. **Tentativa incompatível:** o registry deve recusar (HTTP 409). Guarde a saída para o relatório.
2. Registre a v2. Os serviços usam a última versão registrada quando sobem, então basta reiniciá-los.
3. Reinicie os serviços **consumidores primeiro**, produtores depois. O `LogEvento` continua na v1.

In [10]:
incompativel = json.loads(Path("schemas/PedidoEvento.v2.avsc").read_text(encoding="utf-8"))
for campo in incompativel["fields"]:
    if campo["name"] == "canal_venda":
        campo.pop("default")                      # campo novo SEM valor padrão: quebra o BACKWARD
codigo, resposta = registry("POST", f"/subjects/{SUBJECTS[0]}/versions", {"schema": json.dumps(incompativel)})
print("HTTP", codigo, resposta)
assert codigo == 409, "O registry deveria ter recusado esta versão!"

HTTP 409 {'error_code': 409, 'message': 'Schema being registered is incompatible with an earlier schema for subject "pedidos-validados-value", details: [{errorType:\'READER_FIELD_MISSING_DEFAULT_VALUE\', description:\'The field \'canal_venda\' at path \'/fields/8\' in the new schema has no default value and is missing in the old schema\', additionalInfo:\'canal_venda\'}, {oldSchemaVersion: 1}, {oldSchema: \'{"type":"record","name":"PedidoEvento","namespace":"br.unifal.dce540","doc":"Evento de pedido. Usado nos topicos pedidos-validados, pedidos-confirmados e pedidos-recusados.","fields":[{"name":"pedido_id","type":"string"},{"name":"cliente_id","type":"string"},{"name":"sku","type":"string"},{"name":"quantidade","type":"int"},{"name":"valor","type":"double"},{"name":"ts_evento","type":{"type":"long","logicalType":"timestamp-millis"}},{"name":"etapa","type":["null","string"],"default":null},{"name":"motivo","type":["null","string"],"default":null}]}\'}, {validateFields: \'false\', compa

In [11]:
for s in SUBJECTS:
    print(s, registrar(s, "schemas/PedidoEvento.v2.avsc"))

pedidos-validados-value (200, {'id': 3})
estoque-reservado-value (200, {'id': 3})
pedidos-confirmados-value (200, {'id': 3})
pedidos-recusados-value (200, {'id': 3})


In [12]:
# Com a v2 registrada: reinício na ordem consumidores -> produtores.
ordem = ["metricas", "pagamento", "estoque", "pedidos"]
for nome in ordem:
    parar(nome)
for nome in ordem:
    iniciar(nome)
time.sleep(5)
status()

metricas: encerrado normalmente


pagamento: encerrado normalmente


estoque: encerrado normalmente


pedidos: encerrado normalmente
metricas: iniciado (pid 370513)
pagamento: iniciado (pid 370514)
estoque: iniciado (pid 370515)
pedidos: iniciado (pid 370516)


pedidos    rodando (pid 370516)
estoque    rodando (pid 370515)
pagamento  rodando (pid 370514)
metricas   rodando (pid 370513)


## Etapa 6 - Lote 2

In [13]:
ingestar("entrada/pedidos_lote2.csv", "pedidos", "sku", ["--campo-timestamp", "timestamp"])
esperar_lag_zero()

Tópico 'pedidos' já existia (partições não alteradas).

--- Resumo da ingestão ---
Arquivo:            entrada/pedidos_lote2.csv
Tópico:             pedidos
Mensagens enviadas: 43
Confirmadas:        43
Falhas:             0
Por partição:       {0: 10, 1: 12, 2: 12, 3: 9}
Tempo:              0.13s



19:42:26 {'pedidos': 43, 'estoque': 0, 'pagamento': 0, 'metricas': 0}


19:42:29 {'pedidos': 0, 'estoque': 0, 'pagamento': 0, 'metricas': 0}


19:42:32 {'pedidos': 0, 'estoque': 0, 'pagamento': 0, 'metricas': 0}


19:42:35 {'pedidos': 0, 'estoque': 0, 'pagamento': 0, 'metricas': 0}


19:42:39 {'pedidos': 0, 'estoque': 0, 'pagamento': 0, 'metricas': 0}
Grupos em dia.


True

## Etapa 7 - Encerrar, gravar a saída e verificar

In [14]:
for nome in ["pedidos", "pagamento", "estoque", "metricas"]:
    parar(nome)

# 1) saída dos pedidos e dos logs, a partir dos tópicos (escritor fornecido no kit)
r = subprocess.run([sys.executable, "escritor_saida.py"], capture_output=True, text=True, cwd=RAIZ,
                   env=dict(os.environ, KAFKA_BOOTSTRAP=BOOTSTRAP, SCHEMA_REGISTRY_URL=REGISTRY))
print(r.stdout or r.stderr)

# 2) estoque final e métricas, exportados pelos seus serviços
for exportador in ["servicos/estoque.py", "servicos/metricas.py"]:
    r = subprocess.run([sys.executable, exportador, "--exportar"],
                       capture_output=True, text=True, cwd=RAIZ,
                       env=dict(os.environ, KAFKA_BOOTSTRAP=BOOTSTRAP, SCHEMA_REGISTRY_URL=REGISTRY))
    print(exportador, "->", r.stdout or r.stderr)

r = subprocess.run([sys.executable, "verificar.py", "--saida", "saida", "--gabarito", "gabarito"],
                   capture_output=True, text=True, cwd=RAIZ)
print(r.stdout)
print("verificar.py terminou com código", r.returncode, "(0 = tudo certo)")

pedidos: encerrado normalmente


pagamento: encerrado normalmente


estoque: encerrado normalmente


metricas: encerrado normalmente


Saída gravada em /home/helo/kafka-pedidos/saida
  pedidos_confirmados.jsonl      80 linhas  (tópico pedidos-confirmados)
  pedidos_recusados.jsonl        15 linhas  (tópico pedidos-recusados)
  pedidos_dlq.jsonl               5 linhas  (tópico pedidos-dlq)
  logs_eventos.jsonl            301 linhas  (tópico logs-eventos)



servicos/estoque.py -> saida/estoque_final.csv: 16 skus



servicos/metricas.py -> saida/metricas.csv: 77 linhas

Verificando saida/ contra gabarito/

  [OK] pedidos_confirmados.jsonl: 80 linhas (esperado 80 pedidos)
  [OK] pedidos_recusados.jsonl: 15 linhas (esperado 15 pedidos)
  [OK] pedidos_dlq.jsonl: 5 linhas (esperado 5 pedidos)
  [OK] estoque_final.csv: 16 skus
  [OK] consistência: cada pedido em um único resultado
  [OK] logs_eventos.jsonl: 296 eventos distintos (esperado 296)
  [OK] metricas.csv: 77 linhas

Resultado: 7/7 verificações corretas.

verificar.py terminou com código 0 (0 = tudo certo)


## Encerramento (opcional)

Derruba os contêineres. Rode `ver_log("estoque")` (e os demais) se algo falhou.

In [15]:
# subprocess.run(["docker", "compose", "down"], cwd=RAIZ)